In [1]:
from pytential.sympy_pytential import sympy_pytential
from pytential.reduce import min_pytential
import numpy as np
from sympy import log, symbols
import plotly.graph_objects as go
from plotly.subplots import make_subplots

This file demonstrates how to create, combine, and manipulate pytentials.  Equibilriation is used to reduce free variables. 

# Create phases

Create two binary ideal solutions, using the penalization method to affect the lattice constraint via the elastic energy. 

In [2]:
c0a, c1a, c0b, c1b, Va, Vb = symbols('c0a, c1a, c0b, c1b, Va, Vb')

In [3]:
RT = 8.134*300
kappa = 100000
fa_sp = c0a*RT*(2+log(c0a/(c0a+c1a))) + c1a*RT*(0+log(c1a/(c0a+c1a)))+(c0a+c1a)*kappa/2*(log(Va/(c0a+c1a)))**2
fb_sp = c0b*RT*(0+log(c0b/(c0b+c1b))) + c1b*RT*(1+log(c1b/(c0b+c1b)))+(c0b+c1b)*kappa/2*(log(Vb/(c0b+c1b)))**2

In [4]:
# Build the pytentials from the sympy expressions
fa = sympy_pytential(fa_sp)
fb = sympy_pytential(fb_sp)
print(fa)

x = ['Va', 'c0a', 'c1a']

f(x) = 2440.2*c0a*(log(c0a/(c0a + c1a)) + 2) + 2440.2*c1a*log(c1a/(c0a + c1a)) + (50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))**2

f'(x)= [2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/Va, -2440.2*c1a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c0a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 50000*log(Va/(c0a + c1a))**2 + 2440.2*log(c0a/(c0a + c1a)) + 4880.4 - 2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/(c0a + c1a), -2440.2*c0a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c1a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 50000*log(Va/(c0a + c1a))**2 + 2440.2*log(c1a/(c0a + c1a)) - 2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/(c0a + c1a)]

f"(x)= [[-2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/Va**2 + 2*(50000*c0a + 50000*c1a)/Va**2, 100000*log(Va/(c0a + c1a))/Va - 2*(50000*c0a + 50000*c1a)/(Va*(c0a + c1a)), 100000*log(Va/(c0a + c1a))/Va - 2*(50000*c0a + 50000*c1a)/(Va*(c0a + c1a))], [100000*log(Va/(c0a + c1a))/Va - 2*(50000*c0a + 50000*c1a)/(Va*(c0a + c1a)), -2440.2*c0a/(c0a + c1a)**2 + 2440.2*c1a/

The elastic strain relaxes the lattice constraint. For reference, we can visualize $f^a$ and $f^b$ assuming the lattice constraint still holds; $c_0^a+c_1^a = V^a = 1$. 

In [5]:
x_values = np.linspace(0.001, .999, 100)
Fa = fa(c0a=x_values, c1a=1-x_values, Va = 1)
Fb = fb(c0b=x_values, c1b=1-x_values, Vb = 1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_values, y=Fa, mode='lines', name='fa'))
fig.add_trace(go.Scatter(x=x_values, y=Fb, mode='lines', name='fb'))
fig.update_layout(
    xaxis_title='x',
    yaxis_title='Energy',
    title='fa and fb',
    legend_title='Function'
)
fig.show()

Now we examine the complete space of $f^a$ and $f^b$ and show the previous curves as the valley in the surface: 

In [6]:
# Create meshgrid
X, Y = np.meshgrid(x_values, x_values)

# Calculate Z values for fa and fb
Za = fa(Va=1, c0a=X.ravel(), c1a=Y.ravel()).reshape(X.shape)
Zb = fb(Vb=1, c0b=X.ravel(), c1b=Y.ravel()).reshape(X.shape)

# Line values for the constraint c0a + c1a = 1 (i.e., c1a = 1 - c0a)
fa_line = Fa
fb_line = Fb

# Create subplots for side-by-side surfaces
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('fa(Va=1)', 'fb(Vb=1)'),
    specs=[[{'type': 'surface'}, {'type': 'surface'}]]
)

# Add surface traces
fig.add_trace(go.Surface(z=Za, x=x_values, y=x_values, name='fa surface', showscale=False), row=1, col=1)
fig.add_trace(go.Surface(z=Zb, x=x_values, y=x_values, name='fb surface', showscale=False), row=1, col=2)

# Add line traces on top of each surface
fig.add_trace(go.Scatter3d(x=x_values, y=1-x_values, z=fa_line,
    mode='lines', line=dict(color='red', width=6), name='fa line'), row=1, col=1)

fig.add_trace(go.Scatter3d(x=x_values, y=1-x_values, z=fb_line,
    mode='lines', line=dict(color='blue', width=6), name='fb line'), row=1, col=2)

# Update layout for both subplots
fig.update_layout(
    title='Side-by-Side Surface Plots of fa and fb with Constraint Line',
    scene1=dict(xaxis_title='c0a', yaxis_title='c1a', zaxis_title='fa'),
    scene2=dict(xaxis_title='c0b', yaxis_title='c1b', zaxis_title='fb'),
)

fig.show()

# Combine functions

We now combine both functions into a composite pytential, and add constraints for the total of each species. 

Note the pytential takes c0 and c1 as arguments even though they only appear in the constraints. 

In [7]:
f = fa+fb
c0, c1 = symbols('c0, c1')
f = f.add_constraints_sym([c0a+c0b-c0, c1a+c1b-c1, Va+Vb-1]) 
print(f)

x = ['Va', 'Vb', 'c0', 'c0a', 'c0b', 'c1', 'c1a', 'c1b']

f(x) = 2440.2*c0a*(log(c0a/(c0a + c1a)) + 2) + 2440.2*c0b*log(c0b/(c0b + c1b)) + 2440.2*c1a*log(c1a/(c0a + c1a)) + 2440.2*c1b*(log(c1b/(c0b + c1b)) + 1) + (50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))**2 + (50000*c0b + 50000*c1b)*log(Vb/(c0b + c1b))**2

f'(x)= [2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/Va, 2*(50000*c0b + 50000*c1b)*log(Vb/(c0b + c1b))/Vb, 0, -2440.2*c1a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c0a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 50000*log(Va/(c0a + c1a))**2 + 2440.2*log(c0a/(c0a + c1a)) + 4880.4 - 2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/(c0a + c1a), -2440.2*c1b/(c0b + c1b) + 2440.2*(c0b + c1b)*(-c0b/(c0b + c1b)**2 + 1/(c0b + c1b)) + 50000*log(Vb/(c0b + c1b))**2 + 2440.2*log(c0b/(c0b + c1b)) - 2*(50000*c0b + 50000*c1b)*log(Vb/(c0b + c1b))/(c0b + c1b), 0, -2440.2*c0a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c1a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 50000*log(Va/(c0a + c1a))**2 + 2440.2*log(c1a/(c0a + c1a)) - 2*(50000*c0

## Reduce dimensionality through minimization

$f(V^a, V^b, c_0^a,c_1^a,c_0^b,c_1^b, c_0, c_1)$ is now a function off all the relevant variables but to be useful we must reduce the dimensionality through applying constraints and minimization. 

We can now explore the minimizer capabilities. If we minimize over volume, we will find the lowest common tangent. If we keep it fixed at 0 or 1 we will recover the end members. 

In [8]:
f_min_V = min_pytential(f, ['c0', 'c1', 'Va'])
f_min = min_pytential(f, ['c0', 'c1'])

In [9]:
x_values2 = np.linspace(0.001, .999, 10)
ym = f_min(c0=x_values2, c1 = 1-x_values2)
ymV = f_min_V(c0=x_values2, c1 = 1-x_values2, Va= .999)

In [10]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_values, y=Fa, mode='lines', name='fa', line=dict(color='green')))
fig.add_trace(go.Scatter(x=x_values, y=Fb, mode='lines', name='fb', line=dict(color='red')))
fig.add_trace(go.Scatter(x=x_values2, y=ym, mode='lines', name='f_min'))
fig.add_trace(go.Scatter(x=x_values2, y=ymV, mode='markers', name='f_min_V'))
fig.update_layout(
    xaxis_title='x',
    yaxis_title='Energy',
    title='fa and fb',
    legend_title='Function'
)
fig.show()

We can see what is happening by looking in 3D.

In [11]:
# Create a meshgrid for the two arguments
X, Y = np.meshgrid(x_values2, x_values2)
Z = f_min_V(c0=X.ravel(), c1 = 1-X.ravel(), Va=Y.ravel()).reshape(X.shape)

In [12]:
fig = go.Figure(data=[go.Surface(z=Z, x=x_values2, y=x_values2)])

# Add a line plot for f_a at Va = 1
fig.add_trace(go.Scatter3d(
    x=x_values, y=0*x_values+1, z=Fa,
    mode='lines', name='fa', line=dict(color='green', width=6)
))
# Add a line plot for f_b at Va = 0
fig.add_trace(go.Scatter3d(
    x=x_values, y=0*x_values, z=Fb,
    mode='lines', name='fb', line=dict(color='red', width=6)
))
# Add labels and title
fig.update_layout(
    scene=dict(xaxis_title="c0", yaxis_title="Va", zaxis_title="f"),
)
fig.show()

Check the landscape of c0 vs c1 at a fixed V

In [13]:
Z = f_min_V(c0=X.ravel(), c1 = Y.ravel(), Va=.5).reshape(X.shape)

<lambdifygenerated-881>:3: RuntimeWarning:

invalid value encountered in log

<lambdifygenerated-1070>:3: RuntimeWarning:

invalid value encountered in log



In [14]:
fig = go.Figure(data=[go.Surface(z=Z, x=x_values2, y=x_values2)])

# Add labels and title
fig.update_layout(
    title="Surface Plot of f_min2",
    scene=dict(
        xaxis_title="c0", yaxis_title="c1", zaxis_title="f_min2",
    ),
)
fig.show()

A convenient way to find the solubility limits (equilibrium state) is to minimize over ca and cb:

In [15]:
f0, y0 = f_min.min_fcn(np.array([[0.5, 0.5]]).T)
print(y0[0])

{'c0': 0.5, 'c1': 0.5, 'Va': 0.28727074958795196, 'Vb': 0.712729250412048, 'c0a': 0.02586322066901766, 'c0b': 0.47413677933098236, 'c1a': 0.2614075282381963, 'c1b': 0.23859247176180376}


Using the expansion point, we can build a pYtential as a quadratic expansion

In [18]:
print(f)

x = ['Va', 'Vb', 'c0', 'c0a', 'c0b', 'c1', 'c1a', 'c1b']

f(x) = 2440.2*c0a*(log(c0a/(c0a + c1a)) + 2) + 2440.2*c0b*log(c0b/(c0b + c1b)) + 2440.2*c1a*log(c1a/(c0a + c1a)) + 2440.2*c1b*(log(c1b/(c0b + c1b)) + 1) + (50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))**2 + (50000*c0b + 50000*c1b)*log(Vb/(c0b + c1b))**2

f'(x)= [2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/Va, 2*(50000*c0b + 50000*c1b)*log(Vb/(c0b + c1b))/Vb, 0, -2440.2*c1a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c0a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 50000*log(Va/(c0a + c1a))**2 + 2440.2*log(c0a/(c0a + c1a)) + 4880.4 - 2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/(c0a + c1a), -2440.2*c1b/(c0b + c1b) + 2440.2*(c0b + c1b)*(-c0b/(c0b + c1b)**2 + 1/(c0b + c1b)) + 50000*log(Vb/(c0b + c1b))**2 + 2440.2*log(c0b/(c0b + c1b)) - 2*(50000*c0b + 50000*c1b)*log(Vb/(c0b + c1b))/(c0b + c1b), 0, -2440.2*c0a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c1a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 50000*log(Va/(c0a + c1a))**2 + 2440.2*log(c1a/(c0a + c1a)) - 2*(50000*c0

In [16]:
fq = f.quadratic_expansion(y0[0])
print(fq)

x = ['Va', 'Vb', 'c0', 'c0a', 'c0b', 'c1', 'c1a', 'c1b']

f(x) = 0.5*Va*(348103.660638964*Va - 348103.661463856*c0a - 348103.661463856*c1a) + 0.000236967400414701*Va + 0.5*Vb*(140305.733394848*Vb - 140305.73326084*c0b - 140305.73326084*c1b) - 9.55114433356956e-5*Vb + 0.5*c0a*(-348103.661463856*Va + 433959.435047906*c0a + 339609.236701449*c1a) - 994.633660667324*c0a + 0.5*c0b*(-140305.73326084*Vb + 142028.608768079*c0b + 136881.992630341*c1b) - 994.639509938521*c0b + 0.5*c1a*(-348103.661463856*Va + 339609.236701449*c0a + 348944.086452894*c1a) - 230.21979953437*c1a + 0.5*c1b*(-140305.73326084*Vb + 136881.992630341*c0b + 147109.473749009*c1b) - 230.219741026327*c1b - 612.429489495872

f'(x)= [348103.660638964*Va - 348103.661463856*c0a - 348103.661463856*c1a + 0.000236967400414701, 140305.733394848*Vb - 140305.73326084*c0b - 140305.73326084*c1b - 9.55114433356956e-5, 0, -348103.661463856*Va + 433959.435047906*c0a + 339609.236701449*c1a - 994.633660667324, -140305.73326084*Vb + 142028.60876